In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.25 Bell's Inequality and the Failure of Local Realism

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.25",
    title="Bell's Inequality and the Failure of Local Realism",
    blurb="The puzzle of the entangled pair, settled by an experiment. Einstein "
    "hoped the randomness of quantum mechanics hid pre-assigned answers the "
    "particles carried all along — a local, realistic world. Bell turned that "
    "hope into a number: any such world obeys a strict bound on how correlated "
    "two distant measurements can be, and quantum mechanics sails past it. We "
    "build the local model and watch it fail, compute the quantum value that "
    "beats it, and find the sharp limit at which even quantum correlations stop.",
    difficulty="advanced",
    estimate="165–205 min",
)

## Notebook overview

This notebook opens the final movement — foundations — by making rigorous the
puzzle planted in [§6.8](bloch-sphere-entanglement.ipynb). There, the Bell state showed perfect correlations in
incompatible measurement bases while each particle alone looked random, and we
asserted that no local hidden-variable theory could reproduce it. Bell turned
that assertion into a **theorem with a number**.

Einstein, Podolsky, and Rosen believed quantum randomness merely reflected
ignorance of "elements of reality" the particles carry all along — hidden
variables — and that nature is **local** (no instantaneous influence at a
distance). Bell showed this belief is *testable*. Send two entangled spins to
distant detectors, each set to one of two angles; from the correlations form the
**CHSH** combination $S=E(a,b)-E(a,b')+E(a',b)+E(a',b')$. Any theory that is both
**local** and **realistic** must obey $|S|\le2$ — a bound we will *derive* in
three lines and, more vividly, *simulate* by building explicit hidden-variable
models and watching their correlations never beat 2. Quantum mechanics, computed
from the entangled state, gives $|S|=2\sqrt2\approx2.83$ at the optimal geometry:
decisively above the bound. And yet it does not reach the algebraic maximum of 4
either — it stops at **Tsirelson's bound** $2\sqrt2$, more correlated than any
classical theory but strictly less than logic alone would allow.

The distinctive computational move here is to *build the classical model and
watch it fail*: a Monte Carlo of local hidden-variable strategies that never
exceeds 2, set against the quantum value that does. We then simulate a real Bell
test by sampling outcomes from the Born rule ([§6.4](stern-gerlach-qubit.ipynb)), and close with an honest,
evenhanded account of what the violation does and does not establish.

> **Interpretive care.** We separate cleanly what is *demonstrated* — quantum
> mechanics violates a bound that every local-realistic theory obeys, and
> experiments confirm it — from what is *interpreted*, which is genuinely
> contested. The honest conclusion is "local realism fails," not "reality is an
> illusion." We state the assumptions, note the theorem rules out *local* (not
> nonlocal) hidden variables, and present the interpretations without taking a
> side.

> **Method specificity.** The spin operators are $\cos\theta\,\sigma_z+\sin\theta\,
> \sigma_x$; the quantum correlator $\langle\psi|A\otimes B|\psi\rangle$ uses
> `numpy.kron` and `numpy.vdot`; the hidden-variable Monte Carlo and the
> Born-rule sampling use `numpy.random.default_rng`.

## Theory in brief

### The EPR question and hidden variables

The Bell state ([§6.8](bloch-sphere-entanglement.ipynb)) gives perfectly correlated outcomes in several measurement
bases while each particle alone is random. EPR argued this means the outcomes
must be pre-determined by hidden variables the particles carry, and that quantum
mechanics — lacking them — is incomplete; nature should be **local** (a
measurement here cannot instantly affect a distant particle) and **realistic**
(outcomes reflect pre-existing values).

```{math}
:label: eq-epr
\text{EPR: outcomes} = A(\text{setting},\lambda),\ B(\text{setting},\lambda)
\quad\text{for a shared hidden variable } \lambda .
```

### The CHSH setup

Two entangled spins fly to Alice and Bob. Alice measures along one of two angles
($a$ or $a'$), Bob along one of two ($b$ or $b'$); each outcome is $\pm1$. Over
many pairs the **correlation** for a setting pair is $E(a,b)=\langle(\text{Alice})
(\text{Bob})\rangle$, and the CHSH quantity is

```{math}
:label: eq-chsh-setup
S = E(a,b) - E(a,b') + E(a',b) + E(a',b') .
```

### The local-realistic bound (Bell / CHSH)

In any local hidden-variable theory each particle carries pre-assigned outcomes
$A(\cdot,\lambda),B(\cdot,\lambda)=\pm1$, with **locality** (Alice's outcome does
not depend on Bob's setting). Then for each $\lambda$

```{math}
:label: eq-chsh-bound
A(a)[B(b)-B(b')] + A(a')[B(b)+B(b')] = \pm2 ,
```

because one bracket vanishes and the other is $\pm2$. Averaging over $\lambda$
gives $|S|\le2$ — the **CHSH inequality**, a constraint every local-realistic
theory obeys {cite}`bell1964,chsh1969`.

### The quantum prediction

For the entangled (singlet) state the correlation is $E(a,b)=\langle\psi|A\otimes
B|\psi\rangle=-\cos(\theta_a-\theta_b)$. At the optimal geometry $a=0,\ a'=\pi/2,\
b=\pi/4,\ b'=3\pi/4$,

```{math}
:label: eq-chsh-quantum
|S| = 2\sqrt2 \approx 2.83 \;>\; 2 .
```

Quantum correlations are stronger than any local theory permits.

### Tsirelson's bound

Remarkably, quantum mechanics does **not** reach the algebraic maximum of 4. It
stops at $2\sqrt2$ — squaring the CHSH observable leaves $4$ plus a commutator
term whose norm is at most $4$ (Nielsen & Chuang pose the computation as
*Tsirelson's inequality* among the Chapter 2 problems):

```{math}
:label: eq-tsirelson
\text{local-realistic} \le 2 \;<\; \text{quantum} \le 2\sqrt2 \;<\; \text{algebraic } 4 .
```

More correlated than any classical theory, strictly less than logically possible
— a limit that is itself a deep feature of the theory.

### What is demonstrated, and what is interpreted

The **demonstrated** fact is that quantum mechanics violates a bound obeyed by
every local-realistic theory, and that experiments confirm the quantum
prediction (Aspect 1982 {cite}`aspect1982`; the loophole-free tests of 2015
{cite}`hensen2015`; the 2022 Nobel Prize to Aspect, Clauser, and Zeilinger).

```{math}
:label: eq-interpretation
\text{violation} \Rightarrow \{\text{locality, realism, measurement independence}\}
\text{ cannot all hold.}
```

What this **means** needs care. The theorem rules out *local* hidden variables
but not *nonlocal* ones (Bohmian mechanics reproduces quantum mechanics by being
explicitly nonlocal), and it does not permit faster-than-light signalling — the
marginals stay random (the no-signalling theorem). The interpretations
(Copenhagen, many-worlds, Bohmian, and others) accept different horns; we present
them without adjudicating. The honest conclusion is "local realism fails," not
"reality is an illusion."

- Reference: Bell {cite}`bell1964`; CHSH {cite}`chsh1969`; Aspect et al.
  {cite}`aspect1982`; the loophole-free experiments {cite}`hensen2015`; Nielsen &
  Chuang {cite}`nielsen_chuang`. Cross-reference [§6.8](bloch-sphere-entanglement.ipynb) (the Bell state,
  entanglement, the puzzle posed), [§6.6](pauli-uncertainty.ipynb) (incompatible observables), [§6.5](postulates.ipynb) (the Born
  rule), [§6.4](stern-gerlach-qubit.ipynb) (Born-rule Monte Carlo), and forward to [§6.26](density-matrix.ipynb) (the density matrix),
  [§6.27](quantum-information.ipynb) (quantum information). Named as horizons: GHZ states, the detection and
  locality loopholes, device-independent quantum cryptography.

---
## Setup

We use the qubit form of CHSH: two spin-$\tfrac12$ particles in the **singlet**
state $|\psi^-\rangle=(|01\rangle-|10\rangle)/\sqrt2$, each measured along an axis
in the $x$–$z$ plane at angle $\theta$, with outcomes the $\pm1$ eigenvalues of the
spin operator. The data are the Pauli matrices of [§6.6](pauli-uncertainty.ipynb), that
singlet state ([§6.8](bloch-sphere-entanglement.ipynb)), the four optimal CHSH angles, and the
measurement observable $A(\theta)=\cos\theta\,\sigma_z+\sin\theta\,\sigma_x$ — a transcription
of its displayed definition, and the given input every result below is written in terms of.
The one instrument is the eigenvector extraction that turns that observable into its two
outcome states, a `numpy.linalg.eigh` call and a column convention. Conventions: angles in
radians; `numpy.kron` puts Alice's qubit first.

The objects this notebook is named for are deliberately absent: you write the quantum
correlation `correlation` in Exercise 1, the local hidden-variable model `lhv_chsh` in
Exercise 3, the CHSH combination `chsh` in Exercise 4, and the Born-rule sampler
`born_sampled_correlation` in Exercise 7.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ecp import draw, validate

# data: the Pauli matrices of §6.6, the two the x–z plane needs
SIGMA_X = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
SIGMA_Z = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)

# data: the singlet (Bell) state |ψ⁻⟩ = (|01⟩ − |10⟩)/√2 of §6.8, Alice's qubit
# first — the specimen state the whole notebook measures, not machinery.
SINGLET = (np.kron([1.0, 0.0], [0.0, 1.0]) - np.kron([0.0, 1.0], [1.0, 0.0])) / np.sqrt(
    2
)

# data: the four optimal CHSH angles.
OPTIMAL = dict(a=0.0, ap=np.pi / 2, b=np.pi / 4, bp=3 * np.pi / 4)


# data: the measurement observable, transcribed straight from its displayed definition
# A(θ) = cos θ σ_z + sin θ σ_x. There is nothing to construct beyond the two terms; it is
# the given input that the correlation, the CHSH combination and the sampler are all
# written in terms of.
def spin_operator(theta):
    r"""Spin observable along angle ``theta`` in the $x$–$z$ plane.

    Returns $A(\theta)=\cos\theta\,\sigma_z+\sin\theta\,\sigma_x$, whose $\pm1$
    eigenvalues are the two measurement outcomes.

    Parameters
    ----------
    theta : float
        Measurement angle (radians).

    Returns
    -------
    numpy.ndarray
        The $2\times2$ Hermitian spin operator.
    """
    return np.cos(theta) * SIGMA_Z + np.sin(theta) * SIGMA_X


# instrument: outcome-state plumbing. Diagonalizing a 2×2 Hermitian matrix and picking
# the two columns off in a fixed order is bookkeeping, not Bell physics; Exercises 7 and 8
# need the ± eigenvectors as an input, and neither is about how they are obtained.
def _outcome_bases(theta):
    r"""Return the $(+1, -1)$ eigenvectors of ``spin_operator(theta)``."""
    w, v = np.linalg.eigh(spin_operator(theta))  # ascending: w = (-1, +1)
    return v[:, 1], v[:, 0]  # (+1 vector, -1 vector)

## Exercise 1 — The entangled pair and its correlations

A source emits two spin-$\tfrac12$ particles in the singlet state
$|\psi^-\rangle=(|01\rangle-|10\rangle)/\sqrt2$ ([§6.8](bloch-sphere-entanglement.ipynb)); Alice measures hers
along $\theta_a$, Bob his along $\theta_b$, each reading off a $\pm1$ eigenvalue of
$A(\theta)=\cos\theta\,\sigma_z+\sin\theta\,\sigma_x$. Both the state and that observable are
given in the Setup. What is not given is the single number this whole notebook turns on: the
**correlation** $E(a,b)=\langle\psi|A(\theta_a)\otimes B(\theta_b)|\psi\rangle$, the average of
the product of the two outcomes. On the two-qubit space the joint observable is a Kronecker
product, `numpy.kron` with Alice's factor first (the [§6.8](bloch-sphere-entanglement.ipynb) index convention), and
the expectation is one `numpy.vdot`, real because the operator is Hermitian. The singlet's
answer is the cosine law $E(a,b)=-\cos(\theta_a-\theta_b)$: perfect anticorrelation at equal
angles, none at $90^\circ$, perfect correlation at $180^\circ$, and — because the singlet is
rotationally invariant — a function of the relative angle alone. These are exactly the
correlations EPR argued must come from values the particles carried all along.

1. Write `correlation(tA, tB, state=SINGLET)`: form $A(\theta_a)\otimes B(\theta_b)$ with
   `numpy.kron` on two `spin_operator` calls, and return the real part of
   `numpy.vdot(state, op @ state)`.
2. Evaluate it on a handful of angle pairs, equal angles and a quarter turn among them.
3. Confirm each value equals $-\cos(\theta_a-\theta_b)$.
4. Scan the relative angle over $[0,\pi]$ and plot the cosine it traces out.

Cite {eq}`eq-epr`, {eq}`eq-chsh-setup`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

The singlet correlation must equal $-\cos(\theta_a-\theta_b)$ for every pair of
angles: anticorrelated at equal settings, and the cosine law in between.

In [ ]:
angles = [(0.0, 0.0), (0.0, np.pi / 4), (np.pi / 3, np.pi / 7), (1.1, -0.4)]
validate.close(
    np.array([correlation(tA, tB) for tA, tB in angles]),
    np.array([-np.cos(tA - tB) for tA, tB in angles]),
    "the singlet correlation is E(a,b) = −cos(θ_a − θ_b)",
    rtol=1e-9,
    atol=1e-9,
)

## Exercise 2 — Deriving the CHSH bound for local realism

In a local hidden-variable theory the particles carry their answers with them: a shared
$\lambda$, fixed at the source, determines $A(a,\lambda),A(a',\lambda),B(b,\lambda),
B(b',\lambda)$, each $\pm1$, and **locality** means Alice's value never depends on Bob's
setting. For a single $\lambda$ the CHSH combination is then the algebraic expression
$S(\lambda)=A(a)[B(b)-B(b')]+A(a')[B(b)+B(b')]$ {eq}`eq-chsh-bound`. Since $B(b)$ and $B(b')$
are each $\pm1$, one bracket vanishes and the other is $\pm2$, so $S(\lambda)=\pm2$ for
*every* $\lambda$ — and averaging over $\lambda$ gives $|\langle S\rangle|\le2$, the CHSH
inequality. That is three lines of algebra, but a deterministic hidden variable fixes only
four signs, so there are just $2^4=16$ possible answer sets: the claim can be checked by
exhaustion rather than believed.

1. Enumerate all 16 sign assignments of $A(a),A(a'),B(b),B(b')$.
2. Evaluate $S(\lambda)$ for each.
3. Confirm every value is $\pm2$, so any average over $\lambda$ obeys $|S|\le2$.

Cite {eq}`eq-chsh-bound`.

In [ ]:
# (solution hidden on the public site)


### Validation 2

For every pre-assignment of outcomes, $S(\lambda)=\pm2$; therefore any average
over $\lambda$ satisfies the CHSH bound $|S|\le2$.

In [ ]:
validate.check(
    np.all(np.abs(S_lambda) == 2),
    "S(λ) = ±2 for every hidden-variable assignment, so any local-realistic "
    "theory obeys the CHSH bound |S| ≤ 2",
)

## Exercise 3 — Simulating a local hidden-variable model

The exhaustion argument of Exercise 2 is airtight, but it is worth seeing the classical world
actually try and fail. That is the distinctive computational move here: build Einstein's model
and watch it strain against the bound. A concrete local strategy draws a shared hidden variable
$\lambda$ (an angle, uniform on $[0,2\pi)$, fixed at the source and carried by both particles)
and assigns each party a deterministic outcome $\mathrm{sign}\cos(\lambda-\text{offset})$ that
depends on its *own* setting only — never the distant one, which is exactly what locality
forbids. Four per-setting offsets specify one strategy; drawing them at random samples the
space of strategies. The correlations are then Monte Carlo averages of the outcome products
over many $\lambda$, and $S$ is assembled from the four of them as usual. Randomizing the
offsets buys nothing that convexity does not already rule out — a random strategy is a mixture
of the 16 deterministic ones, and a mixture cannot beat its best member — but the histogram
makes the ceiling visible where the algebra only asserts it.

1. Write `lhv_chsh(rng, n, off)`, returning one local hidden-variable CHSH value: draw $n$
   shared $\lambda$ with `numpy.random.default_rng`, form the four local outcome arrays
   $\mathrm{sign}\cos(\lambda-\text{off}[i])$, and combine their pairwise means into $S$.
   Draw the four offsets at random when none are supplied. **Write this one yourself** — the
   implementation is the lesson.
2. Run it over hundreds of random strategies and report the largest $|S|$ reached.
3. Try the strategy whose offsets are the quantum measurement angles, the most tempting
   classical imitation of the quantum arrangement.
4. Confirm $|S|$ never beats 2, up to Monte Carlo noise.

Cite {eq}`eq-chsh-bound`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

Over hundreds of local hidden-variable strategies, the simulated CHSH value must
stay at or below 2 (within Monte Carlo noise) — no local model exceeds the bound.

In [ ]:
validate.check(
    lhv_values.max() <= 2.02,
    "the simulated local hidden-variable CHSH value stays ≤ 2 over hundreds of "
    "strategies — no local model exceeds the CHSH bound",
)

## Exercise 4 — The quantum violation

Everything is now in place for the confrontation. The quantity Bell bounded is the four-term
combination $S=E(a,b)-E(a,b')+E(a',b)+E(a',b')$ {eq}`eq-chsh-setup` — one minus sign among
four correlations, and no local theory can push its magnitude past 2. Quantum mechanics
supplies the four correlations from the singlet, and the geometry that extracts the most from
them is the equally-fanned one, $a=0,\ a'=\pi/2,\ b=\pi/4,\ b'=3\pi/4$ (the Setup's `OPTIMAL`),
where each of the four relative angles is $45^\circ$ or $135^\circ$ and every term contributes
$1/\sqrt2$ in magnitude. The result is $|S|=2\sqrt2\approx2.83$: past the bound, and past it
by a margin no experimental care could explain away. The singlet is anticorrelated, so the
signed value comes out $-2\sqrt2$; the physical statement is the CHSH one, $|S|\le2$, and it is
the magnitude that breaks it.

1. Write `chsh(state, a, ap, b, bp)`, returning $E(a,b)-E(a,b')+E(a',b)+E(a',b')$ from four
   calls to the `correlation` you wrote in Exercise 1.
2. Evaluate it on the singlet at the optimal angles.
3. Confirm $|S|=2\sqrt2$, and report by how much the local-realistic bound is exceeded.

Cite {eq}`eq-chsh-quantum`.

In [ ]:
# (solution hidden on the public site)


### Validation 4

At the optimal geometry the quantum CHSH magnitude must equal $2\sqrt2$,
decisively above the local-realistic bound of 2.

In [ ]:
validate.close(
    abs(S_quantum),
    2 * np.sqrt(2),
    "quantum mechanics gives |S| = 2√2 ≈ 2.83, violating the local-realistic bound of 2",
    rtol=1e-6,
)

## Exercise 5 — The angle scan

One number at one special geometry invites the suspicion that the violation is a knife-edge —
a coincidence of four carefully chosen angles that a real apparatus, with its imperfect
alignment, would miss. The way to settle that is to vary the geometry. The optimal
arrangement belongs to a one-parameter family, $a=0,\ a'=2\varphi,\ b=\varphi,\ b'=3\varphi$:
four settings equally fanned by $\varphi$, with the optimum at $\varphi=\pi/4$, the
$22.5^\circ$ spacing. Sweeping $\varphi$ across the family traces $|S(\varphi)|$ from the
classical region up through the bound and back, and the fraction of the sweep that sits above
2 measures how forgiving the effect is.

1. Scan $\varphi$ over $[0,\pi/2]$, evaluating $|S(\varphi)|$ on that family with the `chsh`
   you wrote in Exercise 4.
2. Locate the peak and confirm it is $2\sqrt2$ at $\varphi=\pi/4$.
3. Report the fraction of the scan with $|S|>2$ — the violation is robust, not fine-tuned.

Cite {eq}`eq-chsh-quantum`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

The scanned quantum CHSH value must peak at $2\sqrt2$ and exceed 2 over a broad
range of geometries (not a single fine-tuned point).

In [ ]:
validate.close(
    S_scan.max(),
    2 * np.sqrt(2),
    "the CHSH value peaks at 2√2 for the optimal geometry and exceeds 2 broadly",
    rtol=1e-3,
)
validate.check(
    frac_above > 0.5, "the quantum violation |S|>2 holds over most of the scan (robust)"
)

## Exercise 6 — Tsirelson's bound

The scan of Exercise 5 has a second lesson hidden in it, and it points the other way. Nothing
in arithmetic stops $S$ at $2\sqrt2$: each of the four correlations lies in $[-1,1]$, so if
they could be chosen freely the combination would reach 4, and a hypothetical super-quantum
theory obeying only no-signalling would do exactly that. Quantum mechanics does not. Squaring
the CHSH observable leaves $4$ plus a commutator term whose norm is at most $4$, which caps
$|S|$ at $2\sqrt2$ — **Tsirelson's bound** {eq}`eq-tsirelson`, and the scan saturates it
without ever crossing it. The ordering that results, local-realistic $\le2<$ quantum
$\le2\sqrt2<$ algebraic 4, says quantum correlations are stronger than any classical theory
permits and strictly weaker than logic alone would allow. *Why* nature stops precisely there
is still an open research question.

1. Take the maximum of the quantum scan from Exercise 5 and compare it to $2\sqrt2$.
2. Confirm it never exceeds that bound.
3. Set the three ceilings — 2, $2\sqrt2$, and the algebraic 4 — side by side.

Cite {eq}`eq-tsirelson`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

The quantum CHSH value must never exceed $2\sqrt2$: quantum mechanics respects
Tsirelson's bound, sitting strictly between the classical bound and the algebraic
maximum.

In [ ]:
validate.check(
    S_scan.max() <= 2 * np.sqrt(2) + 1e-6,
    "the quantum CHSH value never exceeds 2√2 (Tsirelson): local ≤2 < quantum ≤2√2 < 4",
)

## Exercise 7 — Simulating the actual experiment *(student)*

Every $S$ computed so far has been an expectation value — an exact average over an infinite
ensemble, which no laboratory has. A real Bell test counts coincidences: pair after pair,
each arriving with one of the four joint outcomes $(\pm1,\pm1)$, tallied at four analyzer
settings until the four correlations are known well enough. Reproducing that means sampling
rather than averaging. The joint outcome probabilities come from the Born rule
([§6.5](postulates.ipynb)): with $|a_\pm\rangle$ and $|b_\pm\rangle$ the two parties' outcome eigenstates
(the Setup's `_outcome_bases`), the probability of the pair $(s_a,s_b)$ is
$|\langle a_{s_a}b_{s_b}|\psi\rangle|^2$, and the correlation is the sample mean of the
product $s_as_b$ over $n$ draws — the Born-rule Monte Carlo of [§6.4](stern-gerlach-qubit.ipynb) lifted from one
qubit to two. Finite $n$ means a statistical error $\sim1/\sqrt n$, so the honest presentation
is not one number but a convergence: the estimate settling onto $2\sqrt2$ as the counts grow,
with the classical bound receding by more and more standard errors. That is what Aspect
measured in 1982 and what the loophole-free tests of 2015 measured with the loopholes shut.

1. Write `born_sampled_correlation(tA, tB, n, rng, state=SINGLET)`: build the four joint
   outcome probabilities from `_outcome_bases` and `numpy.kron`, normalize them, draw $n$
   outcome pairs with `rng.choice`, and return the mean of the outcome products.
   **Write this one yourself** — the implementation is the lesson.
2. Estimate the four correlations at the optimal settings and assemble $|S|$ from them, for a
   ladder of sample sizes.
3. Confirm $|S|\approx2\sqrt2$ within statistical error at the largest sample size.

Cite {eq}`eq-chsh-quantum`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

Born-rule sampling of the entangled state must reproduce the CHSH violation:
$|S|\approx2\sqrt2$ within statistical error, and comfortably above 2.

In [ ]:
validate.close(
    S_sampled,
    2 * np.sqrt(2),
    "Born-rule sampling of the entangled state reproduces the CHSH violation (≈2√2)",
    atol=0.05,
)

## Exercise 8 — What must give way *(student / interpretation)*

The violation is a fact; what it costs is a question. Bell's derivation rests on three
assumptions, and the experiment refutes their conjunction, not any one of them: **locality**
(an outcome depends only on the local setting and the shared past), **realism** (outcomes are
determined by pre-existing values), and **measurement independence** (the settings are chosen
freely, independently of $\lambda$). At least one must fail. What fails is *local* hidden
variables, not hidden variables as such — Bohmian mechanics reproduces every quantum
prediction by being explicitly nonlocal, and pays for it exactly there. What does *not* fail
is relativity, and the reason is worth checking rather than asserting: the correlations are
invisible to either party alone. Alice's marginal probability of a $+1$ outcome is obtained
by summing the joint Born probabilities over Bob's two outcomes {eq}`eq-interpretation`, and
for the singlet it is $1/2$ whatever Bob does — so nothing Bob chooses can be read off
Alice's data, and no signal passes. Which assumption to surrender is where Copenhagen,
many-worlds and Bohmian mechanics part ways, and this notebook takes no side; the honest
conclusion is "local realism fails," not "reality is an illusion."

1. Write `alice_marginal(tA, tB)`, summing $|\langle a_+b_\pm|\psi\rangle|^2$ over Bob's two
   outcome states to give $P(\text{Alice}=+1)$ with Bob measuring at $\theta_b$.
2. Evaluate it at a fixed $\theta_a$ for several of Bob's settings.
3. Confirm every value is $1/2$: Alice learns nothing about Bob's setting, so the violation
   forbids local realism without permitting faster-than-light communication.

Cite {eq}`eq-interpretation`.

In [ ]:
# (solution hidden on the public site)


### Validation 8

Alice's single-party marginal must be independent of Bob's distant setting
(always $1/2$): the violation defeats local realism but forbids signalling — the
marginals carry no information.

In [ ]:
validate.close(
    np.array(marginals),
    0.5 * np.ones(4),
    "the single-party marginals are 1/2 regardless of the distant setting "
    "(no signalling: the violation forbids local realism, not relativity)",
    atol=1e-9,
)

## Exercise 9 — (Synthesis) A number that decided a debate

For thirty years the question of whether quantum randomness hid a deeper, local,
deterministic layer was thought to be metaphysics, beyond experiment. Bell turned
it into arithmetic: a single number, bounded by 2 for any local-realistic world,
predicted to be $2\sqrt2$ by quantum mechanics. We built the local model and
watched it strain against the bound and fail — no strategy, however clever, beat
2. We computed the quantum value and watched it pass, at $2\sqrt2\approx2.83$. And
we found that quantum mechanics, for all its strangeness, stops short of the
maximum imaginable: more correlated than any classical theory, less than logic
alone would allow. Born-rule sampling reproduced the violation from simulated
coincidence counts, exactly as a real experiment does.

It is worth pausing on how unusual this is. A philosophical dispute between
Einstein and Bohr about the nature of reality was settled — not dissolved,
settled — by a table of measured coincidence counts. Aspect's 1982 experiment,
the loophole-free tests of 2015, and the 2022 Nobel Prize confirm it: the
correlations of the Bell state are real, they defeat local realism, and no
experiment has ever sided against them. What remains open is not *whether*
quantum mechanics is right but *what it means* — and on that, the honest word is
that at least one of locality, realism, or free choice must go, while the
no-signalling theorem keeps the whole thing consistent with relativity. Which
assumption to surrender is where Copenhagen, many-worlds, and Bohmian mechanics
part ways, and this notebook takes no side.

With that, **Movement VI is open** — the foundations, where we ask not how to
compute with quantum mechanics but what it is telling us. The next notebook ([§6.26](density-matrix.ipynb))
steps back to ask what "the state of a subsystem" even means when parts are
entangled like this: the **density matrix**, the proper description of the mixed,
correlated pieces that Bell pairs are made of.

## Notebook summary

- **The singlet correlations** {eq}`eq-chsh-setup`: $E(a,b)=-\cos(\theta_a-
  \theta_b)$, computed as $\langle\psi|A\otimes B|\psi\rangle$ (`numpy.kron`/`vdot`).
- **The CHSH bound** {eq}`eq-chsh-bound`: $S(\lambda)=\pm2$ for every
  pre-assignment, so any local-realistic theory obeys $|S|\le2$ (derived + brute-checked).
- **The local model fails**: hundreds of simulated hidden-variable strategies
  (`numpy.random.default_rng`) never beat 2.
- **The quantum violation** {eq}`eq-chsh-quantum`: $|S|=2\sqrt2\approx2.83$ at the
  optimal angles, exceeding 2 over ~76% of geometries.
- **Tsirelson** {eq}`eq-tsirelson`: quantum $|S|\le2\sqrt2<4$ — strong but not maximal.
- **The experiment, simulated**: Born-rule sampling reproduces $|S|\approx2\sqrt2$;
  the marginals stay $1/2$ (no signalling). The honest conclusion: **local realism
  fails**.

> **What is settled, and what is not.** *Settled:* quantum mechanics violates a
> bound every local-realistic theory obeys, and experiment agrees. *Open:* which
> assumption to give up, and what the violation means. This notebook demonstrates
> the first and lays out the second without adjudicating.

## Outlook

- **The density matrix and mixed states** ([§6.26](density-matrix.ipynb)): the proper description of a
  subsystem of an entangled pair.
- **Quantum information and device-independent cryptography** ([§6.27](quantum-information.ipynb)), where Bell
  violations *certify* security.
- **GHZ states**: a single-measurement, all-or-nothing version of the
  contradiction (a horizon).
- **The loopholes** (detection, locality, freedom-of-choice) and the loophole-free
  experiments (a horizon).
- Cross-reference [§6.8](bloch-sphere-entanglement.ipynb) (the Bell state, entanglement), [§6.6](pauli-uncertainty.ipynb) (incompatible
  observables), [§6.5](postulates.ipynb) (the Born rule), [§6.4](stern-gerlach-qubit.ipynb) (Born-rule Monte Carlo), forward to [§6.26](density-matrix.ipynb),
  [§6.27](quantum-information.ipynb).

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()